# Climate experiments with a radiation code

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/evanwellmeyer/PySoc/blob/main/notebooks/02_perturbation_experiments.ipynb)

In [Notebook 1](https://colab.research.google.com/github/evanwellmeyer/PySoc/blob/main/notebooks/01_how_radiation_works.ipynb)
we looked inside SOCRATES, the radiation scheme of the Isca climate model. Here we use it as a
laboratory. In each activity you **change something** about an atmospheric column (its CO₂, its
temperature, a cloud, the ground beneath it) and **describe how the column responds**. The activities
build up to an estimate of the planet's **climate sensitivity**: how much the surface warms when CO₂
doubles.

| Activity | What you change | What to look at |
|---|---|---|
| 1 | the amount of CO₂ | the OLR, the energy entering each level, the heating rates |
| 2 | CO₂ over a very wide range | how the forcing grows |
| 3 | CO₂, or the brightness of the Sun | the pattern of heating and cooling |
| 4 | the temperature (and humidity) of the column | how much more energy escapes |
| 5 | one layer at a time, using gradients | which layers matter most |
| 6 | a cloud's height, thickness and water | the cloud's effect on energy and heating |
| 7 | the surface albedo and the sun's height | where sunlight is absorbed |
| 8 | CO₂, humidity and convection in a model that finds its own temperatures | the equilibrium climate |

**How to work.** Run each cell once (▶ or Shift+Enter, or *Runtime → Run all*) to show its sliders, tick
boxes and menus. Then move them: the numbers and plots update as you go (**Reset to defaults** puts them
back where they started). Try several values, and write
your description in the *Your notes* cell under the activity (double-click it to type). The experiments
in Activity 8 take several seconds each, so they wait for you to press **Run the model**.

Every activity compares your perturbed column with the same **control** column and shows the response
the same way:
* a table of the energy budget: control, perturbed, and the change;
* the change in **heating rate** at each level (K/day): where the column now warms or cools;
* the change in the **net energy flowing down** through each level (W/m²): positive means extra energy
  is being trapped below that level;
* the change in the **OLR in each longwave band**: which wavelengths respond.

The code in each cell is hidden to keep the page readable. Click **Show code** (or double-click a cell's
title) if you want to read it.

In [ ]:
#@title Setup: run this cell first (it takes about a minute on Colab)
import functools, importlib, os, subprocess, sys, time

if os.path.isdir("../pysoc") and os.path.abspath("..") not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))  # running inside a copy of the repository
try:
    import pysoc
except ImportError:  # on Colab: install PySoc from GitHub
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/evanwellmeyer/PySoc"],
                   check=True)
    importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import torch
from cycler import cycler
from IPython.display import display
from ipywidgets import Button, Checkbox, Dropdown, FloatSlider, IntSlider, Layout, interactive

from pysoc.column import (make_column, liquid_cloud, hydrostatic_heights, saturation_specific_humidity,
                          specific_humidity)
from pysoc.isca import IscaSocrates, GAS_NAMES, CP_AIR, GRAV, RDGAS
from pysoc.spectra import ga7_spectral_files

torch.set_flush_denormal(True)  # avoids a slowdown in float64 on CPUs
torch.set_num_threads(min(4, torch.get_num_threads()))  # one column runs fastest on a few threads
lw_file, sw_file = ga7_spectral_files()  # downloads the SOCRATES GA7 spectral files once
model = IscaSocrates(lw_file, sw_file)

BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948")
INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
LW_COLOR, SW_COLOR, NET_COLOR = BLUE, ORANGE, INK
plt.rcParams.update({
    "figure.dpi": 100, "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "axes.titlecolor": INK, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.labelsize": 10, "font.size": 10,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelcolor": INK2, "ytick.labelcolor": INK2, "lines.linewidth": 2, "legend.frameon": False,
    "axes.prop_cycle": cycler(color=[BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED]),
})


def radiation(col, albedo=0.3, insolation=340.0, coszen=0.5, cloud=None, remove=()):
    """Run SOCRATES on a column (see Notebook 1). Returns a dict of outputs named as in Isca."""
    ids = {name: i for i, name in GAS_NAMES.items()}
    model.config.exclude_gases = frozenset(ids[name] for name in remove)
    try:
        rrsun = insolation / (coszen * model.config.stellar_constant)
        return model(**col, albedo=albedo, coszen=coszen, rrsun=rrsun, delta_t=0.0, **(cloud or {}))
    finally:
        model.config.exclude_gases = frozenset()


def toa_net(out):
    """Net energy into the planet at the top of the atmosphere: absorbed sunlight minus OLR (W/m2)."""
    return out["soc_toa_sw"] - out["soc_olr"]


def net_down(out):
    """Net energy flowing down (longwave + shortwave) through each half level (W/m2)."""
    return -(out["soc_flux_lw"] + out["soc_flux_sw"])


def warmed(col, dT=1.0, fixed_rh=False):
    """A copy of the column with the air and surface warmer by dT (K).

    dT is a number, or one value per layer (then the surface warms like the lowest layer).
    fixed_rh=True raises the water vapour so the relative humidity of every layer stays the same.
    """
    new = dict(col)
    dT = torch.broadcast_to(torch.as_tensor(dT, dtype=col["temp"].dtype), col["temp"].shape)
    new["temp"] = col["temp"] + dT
    new["t_surf"] = col["t_surf"] + dT[..., -1]
    new["z_full"], new["z_half"] = hydrostatic_heights(new["temp"], col["p_half"], col["p_full"])
    if fixed_rh:
        new["q"] = col["q"] * saturation_specific_humidity(new["temp"], col["p_full"]) \
            / saturation_specific_humidity(col["temp"], col["p_full"])
    return new


def pressure_axis(ax, top=0.01):
    ax.set_yscale("log")
    ax.set_ylim(1000, top)
    ax.set_yticks([t for t in (1000, 700, 500, 300, 200, 100, 30, 10, 3, 1, 0.3, 0.1, 0.03, 0.01)
                   if t >= top and (top >= 100 or t not in (700, 500, 200))])
    ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
    ax.get_yaxis().set_minor_formatter(plt.NullFormatter())
    ax.set_ylabel("pressure (hPa)")


def band_label(sp, b):
    lo, hi = sp.wavelength_short[b] * 1e6, sp.wavelength_long[b] * 1e6
    return f"{b + 1}: {lo:.3g}–{hi:.3g} µm" if hi < 1000 else f"{b + 1}: {lo:.3g}–{hi:.0f} µm"


hpa = lambda p: (p / 100).detach().numpy()  # noqa: E731  Pa -> hPa
per_day = lambda rate: (rate * 86400).detach().numpy()  # noqa: E731  K/s -> K/day
lw_spec = model.lw.spectrum.sp

# the control column, which every activity compares against
control = make_column(t_surf=288.0, co2_ppmv=280.0)
o_ctl = radiation(control)
p = hpa(control["p_full"])
p_half = hpa(control["p_half"])
p_half[0] = 0.01  # the top half level is at p = 0; plot it at the top of the axis
TROP = int(np.argmin(np.abs(p_half - 200)))  # half level closest to 200 hPa, our stand-in for the tropopause
F_2X = float(net_down(radiation(make_column(t_surf=288.0, co2_ppmv=560.0)))[TROP] - net_down(o_ctl)[TROP])


def olr_bands(out):
    """Outgoing longwave radiation in each band, all sky (W/m2)."""
    return out["flux_lw_up_band"][..., 0, :]


def budget(o_new, o_ref=None, name="perturbed"):
    """Print the energy budget of the control and perturbed columns, and the change."""
    rows = (("outgoing longwave (OLR)", lambda o: o["soc_olr"]),
            ("sunlight absorbed by the planet", lambda o: o["soc_toa_sw"]),
            ("net energy in at the top", toa_net),
            ("net energy in at 200 hPa", lambda o: net_down(o)[TROP]),
            ("sunlight absorbed at the surface", lambda o: o["soc_surf_flux_sw"]),
            ("longwave from the sky at the surface", lambda o: o["soc_surf_flux_lw_down"]))
    if o_ref is None and o_new is o_ctl:  # just the control
        for label, get in rows:
            print(f"{label:38s}{float(get(o_ctl)):8.1f} W/m²")
        return
    o_ref = o_ctl if o_ref is None else o_ref
    print(f"{'W/m²':38s}{'control':>10s}{name[:12]:>13s}{'change':>10s}")
    for label, get in rows:
        a, b = float(get(o_ref)), float(get(o_new))
        print(f"{label:38s}{a:10.1f}{b:13.1f}{b - a:+10.2f}")


def response_plots(o_new, o_ref=None, top=0.01, title="How the column responds"):
    """Changes in heating rate, in net downward flux, and in OLR band by band."""
    o_ref = o_ctl if o_ref is None else o_ref
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.4), gridspec_kw=dict(width_ratios=[1, 1, 1.25]))
    ax = axes[0]
    ax.axvline(0, color=AXIS, linewidth=1)
    ax.plot(per_day(o_new["tdt_lw"] - o_ref["tdt_lw"]), p, color=LW_COLOR, label="longwave")
    ax.plot(per_day(o_new["tdt_sw"] - o_ref["tdt_sw"]), p, color=SW_COLOR, label="shortwave")
    ax.plot(per_day(o_new["tdt_rad"] - o_ref["tdt_rad"]), p, color=NET_COLOR, label="total")
    pressure_axis(ax, top)
    ax.set_xlabel("change in heating rate (K/day)")
    ax.set_title("Heating and cooling")
    ax.legend(loc="best")
    ax = axes[1]
    ax.axvline(0, color=AXIS, linewidth=1)
    ax.axhline(p_half[TROP], color=MUTED, linewidth=1)
    ax.annotate("200 hPa", (1.0, p_half[TROP]), xycoords=("axes fraction", "data"), xytext=(-4, 3),
                textcoords="offset points", ha="right", va="bottom", fontsize=9, color=INK2)
    ax.plot((net_down(o_new) - net_down(o_ref)).detach().numpy(), p_half, color=INK)
    pressure_axis(ax, top)
    ax.set_ylabel("")
    ax.set_xlabel("change in net energy flowing down (W/m²)")
    ax.set_title("Energy trapped below each level")
    ax = axes[2]
    d_band = (olr_bands(o_new) - olr_bands(o_ref)).detach().numpy()
    y = np.arange(len(d_band))
    ax.barh(y, d_band, height=0.7, color=LW_COLOR)
    ax.axvline(0, color=AXIS, linewidth=1)
    ax.set_yticks(y, [band_label(lw_spec, b) for b in y])
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("change in OLR (W/m²)")
    ax.set_title("OLR, band by band")
    fig.suptitle(title, x=0.01, ha="left", fontsize=12, fontweight="bold", color=INK)
    fig.tight_layout()
    plt.show()


def slider(value, lo, hi, step, label, live=True):
    """A labelled slider. With live=True the plots redraw while it moves; otherwise when it is let go."""
    kind = IntSlider if all(float(v).is_integer() for v in (value, lo, hi, step)) else FloatSlider
    return kind(value=value, min=lo, max=hi, step=step, description=label, continuous_update=live,
                style={"description_width": "240px"}, layout=Layout(width="620px"))


def tickbox(value, label):
    return Checkbox(value=value, description=label, style={"description_width": "240px"}, layout=Layout(width="620px"))


def menu(options, value, label):
    return Dropdown(options=options, value=value, description=label, style={"description_width": "240px"},
                    layout=Layout(width="620px"))


def controls(f, manual=False, **widgets):
    """Like ipywidgets' interact(f, ...), plus a button that puts every control back to its starting value.

    manual=True waits for a "Run the model" button instead of redrawing whenever a control moves.
    """
    resetting = [False]

    @functools.wraps(f)
    def run(**kwargs):
        if not resetting[0]:
            return f(**kwargs)

    ui = interactive(run, {"manual": True, "manual_name": "Run the model"} if manual else {}, **widgets)
    defaults = [(w, w.value) for w in ui.kwargs_widgets]
    reset_button = Button(description="Reset to defaults", icon="undo", layout=Layout(width="170px"))

    def reset(_):
        resetting[0] = True  # move every control back without redrawing for each one...
        try:
            for w, value in defaults:
                w.value = value
        finally:
            resetting[0] = False
        if not manual:
            ui.update()  # ...then redraw once

    reset_button.on_click(reset)
    ui.children = ui.children[:-1] + (reset_button, ui.children[-1])  # the button sits above the plots
    display(ui)

print("Ready.")

## The control column

Every activity starts from the same **control** column: a pre-industrial atmosphere (280 ppm CO₂) over a
surface at 288 K, with global-mean sunlight (340 W/m²), a surface albedo of 0.3 and no clouds. We use
200 hPa as a simple stand-in for the tropopause. This cell shows the control column's profiles and
energy budget.

In [ ]:
#@title The control column
fig, axes = plt.subplots(1, 3, figsize=(11, 4), sharey=True)
axes[0].plot(control["temp"].numpy(), p, color=RED)
axes[0].set_xlabel("temperature (K)")
axes[1].plot(1000 * control["q"].numpy(), p, color=BLUE)
axes[1].set_xscale("log")
axes[1].set_xlabel("water vapour (g/kg)")
axes[2].axvline(0, color=AXIS, linewidth=1)
axes[2].plot(per_day(o_ctl["tdt_lw"]), p, color=LW_COLOR, label="longwave")
axes[2].plot(per_day(o_ctl["tdt_sw"]), p, color=SW_COLOR, label="shortwave")
axes[2].plot(per_day(o_ctl["tdt_rad"]), p, color=NET_COLOR, label="total")
axes[2].set_xlabel("radiative heating (K/day)")
axes[2].legend(loc="lower left")
pressure_axis(axes[0])
for ax in axes[1:]:
    ax.set_ylabel("")
fig.suptitle("The control column", x=0.01, ha="left", fontsize=12, fontweight="bold", color=INK)
fig.tight_layout()
plt.show()
budget(o_ctl)

---
## Activity 1: Change the CO₂

CO₂ has risen from 280 ppm before the industrial revolution to over 420 ppm today. Change the CO₂ while
keeping everything else (temperatures, humidity) fixed. The change in the energy budget that results is
called the **radiative forcing**.

Start with 560 ppm (double the control), then try 420, 1120, 140 and 0 ppm. Tick *zoom in on the
troposphere* to look at the lower atmosphere in detail.

In [ ]:
#@title Activity 1: change the CO₂
def activity_1(co2_ppmv, zoom_to_troposphere):
    o_new = radiation(make_column(t_surf=288.0, co2_ppmv=co2_ppmv))
    budget(o_new, name=f"{co2_ppmv} ppm")
    response_plots(o_new, top=100 if zoom_to_troposphere else 0.01, title=f"CO₂ changed from 280 to {co2_ppmv} ppm")


controls(activity_1, co2_ppmv=slider(560, 0, 2240, 10, "CO₂ (ppm)"),
         zoom_to_troposphere=tickbox(False, "zoom in on the troposphere"));

**Describe how the column responds**
1. How does the OLR change when you double the CO₂? How does the net energy entering the column at
   200 hPa compare with the change at the top of the atmosphere?
2. Look at the heating and cooling panel. Where does the column warm, where does it cool, and by roughly
   how much?
3. Which bands respond most? Does the window band (5) respond at all?
4. Compare 280 → 560 ppm with 280 → 1120 ppm and with 280 → 140 ppm. How does the size of the response
   change?

**Explain:** the extra energy trapped below 200 hPa is almost twice the change at the top. What is the
stratosphere doing with the difference?

<details><summary>Hint</summary>

Look at the heating panel above 200 hPa. More CO₂ in the stratosphere means more emission from the
stratosphere, both upward and downward.
</details>

*Your notes:*

---
## Activity 2: CO₂ over a very wide range

PySoc can run many columns at once. This cell builds columns whose CO₂ doubles from one to the next,
runs them all in **one** call, and plots the forcing at 200 hPa against the amount of CO₂. The grey line
is a widely used formula, 5.35 ln(C/280) W/m² (Myhre et al., 1998), derived from calculations that
include clouds.

Change the starting amount and the number of doublings, and switch between a logarithmic and an
ordinary CO₂ axis.

In [ ]:
#@title Activity 2: CO₂ over a very wide range
def activity_2(lowest_co2_ppmv, number_of_doublings, logarithmic_co2_axis):
    co2_values = float(lowest_co2_ppmv) * 2.0 ** torch.arange(number_of_doublings + 1, dtype=torch.float64)
    o_many = radiation(make_column(t_surf=288.0, co2_ppmv=co2_values))  # one column per CO2 value, one call
    forcing = (net_down(o_many)[:, TROP] - net_down(o_ctl)[TROP]).numpy()
    for c, f in zip(co2_values.tolist(), forcing):
        print(f"{c:9.1f} ppm   forcing at 200 hPa {f:+7.2f} W/m²")
    print("added by each doubling:", np.round(np.diff(forcing), 2), "W/m²")

    fig, ax = plt.subplots(figsize=(6.8, 4.2))
    ax.axhline(0, color=AXIS, linewidth=1)
    ax.plot(co2_values.numpy(), forcing, marker="o", markersize=7, color=INK, label="PySoc, clear sky, 200 hPa")
    smooth = np.geomspace(co2_values[0].item(), co2_values[-1].item(), 200)
    ax.plot(smooth, 5.35 * np.log(smooth / 280.0), color=MUTED, linewidth=1.5, label="5.35 ln(C/280)")
    if logarithmic_co2_axis:
        ax.set_xscale("log", base=2)
        ax.set_xticks(co2_values.tolist(), [f"{c:g}" for c in co2_values.tolist()])
    ax.set_xlabel("CO₂ (ppm" + (", log scale)" if logarithmic_co2_axis else ")"))
    ax.set_ylabel("forcing at 200 hPa (W/m²)")
    ax.set_title("Forcing as CO₂ doubles again and again")
    ax.legend(loc="upper left")
    fig.tight_layout()
    plt.show()


controls(activity_2, lowest_co2_ppmv=slider(35, 1, 1000, 1, "lowest CO₂ (ppm)"),
         number_of_doublings=slider(7, 1, 10, 1, "number of doublings"),
         logarithmic_co2_axis=tickbox(True, "logarithmic CO₂ axis"));

**Describe how the forcing grows**
1. On the logarithmic axis, what shape does the forcing make? What does each doubling add?
2. Switch to the ordinary axis. Describe the shape now. Does the first 100 ppm of CO₂ matter more or
   less than the last 100 ppm?
3. Start from 1 ppm, or run up to very high CO₂ (10 doublings from 35 ppm). Does the pattern hold at the
   extremes?
4. How does our clear-sky column compare with the formula?

**Explain:** why doesn't the forcing grow in proportion to the amount of CO₂? Use the band-by-band plot
from Activity 1.

<details><summary>Hint</summary>

At the centre of the 15 µm band the atmosphere is already opaque, so extra CO₂ there only moves the
emission level up a little. Most of the change comes from the band's edges, where absorption is weaker.
</details>

*Your notes:*

---
## Activity 3: CO₂ or the Sun?

The Sun's output also varies. Could a brighter Sun explain global warming instead of CO₂? Radiation
gives us a way to tell them apart. This cell compares how extra CO₂ and a brighter Sun change the
**heating rate** at every level. The right-hand panel zooms in on the troposphere.

Adjust the Sun's brightness until its change in net energy at the top of the atmosphere matches the
CO₂ change, so the two are compared at the same strength.

In [ ]:
#@title Activity 3: CO₂ or the Sun?
def activity_3(co2_ppmv, sun_brighter_percent):
    o_co2 = radiation(make_column(t_surf=288.0, co2_ppmv=co2_ppmv))
    o_sun = radiation(control, insolation=340.0 * (1 + sun_brighter_percent / 100))
    print(f"change in net energy in at the top:   CO₂ {float(toa_net(o_co2) - toa_net(o_ctl)):+.2f} W/m²,"
          f"   Sun {float(toa_net(o_sun) - toa_net(o_ctl)):+.2f} W/m²")
    d_co2, d_sun = per_day(o_co2["tdt_rad"] - o_ctl["tdt_rad"]), per_day(o_sun["tdt_rad"] - o_ctl["tdt_rad"])
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
    for ax, top in zip(axes, (0.01, 100)):
        ax.axvline(0, color=AXIS, linewidth=1)
        ax.plot(d_co2, p, color=INK, label=f"CO₂ {co2_ppmv} ppm")
        ax.plot(d_sun, p, color=SW_COLOR, label=f"Sun {sun_brighter_percent:+.1f}%")
        pressure_axis(ax, top)
        ax.set_xlabel("change in heating rate (K/day)")
    axes[0].set_title("Whole atmosphere")
    axes[0].legend(loc="lower left")
    axes[1].set_title("Troposphere")
    axes[1].set_ylabel("")
    lim = 1.2 * max(np.abs(d_co2[p > 100]).max(), np.abs(d_sun[p > 100]).max(), 1e-3)
    axes[1].set_xlim(-lim, lim)
    fig.tight_layout()
    plt.show()


controls(activity_3, co2_ppmv=slider(560, 280, 1120, 10, "CO₂ (ppm)"),
         sun_brighter_percent=slider(1.0, -3.0, 3.0, 0.1, "Sun brighter by (%)"));

**Describe the two patterns**
1. With the two changes at the same strength at the top, describe how each one changes the heating of
   the stratosphere, and of the troposphere.
2. Try a dimmer Sun (a negative percentage). Is the pattern simply reversed?
3. Weather balloons and satellites show the troposphere warming and the stratosphere *cooling* since the
   1970s. Which of your two experiments does that look like?

**Explain:** why does more CO₂ cool the stratosphere? (Hint: in the stratosphere CO₂ both absorbs radiation
from below and emits its own. Which of the two gains more when CO₂ increases?)

*Your notes:*

---
## Activity 4: Warm the column

A forcing makes the planet gain energy, so it warms. A warmer planet radiates more, which pushes back.
The **feedback parameter** λ is how much more energy escapes (in W/m²) per degree of surface warming.
The equilibrium warming for a forcing F is then roughly ΔT ≈ F / λ.

This cell warms the whole column (air and surface) and measures λ. You can choose:
* *keep the relative humidity fixed*: unticked, the amount of water vapour stays fixed; ticked, the air gains
  water vapour so its relative humidity stays the same (about 7% more water per K, as in the real atmosphere);
* *extra warming of the upper troposphere*: extra warming that grows with height through the troposphere, as
  in the real tropics, where the upper troposphere warms more than the surface. The stratosphere always warms
  by the main *warming* value.

In [ ]:
#@title Activity 4: warm the column
def activity_4(warming_K, keep_relative_humidity, extra_upper_troposphere_warming_K):
    p_full = control["p_full"]
    shape = torch.where(p_full > 200e2, (1e5 - p_full) / (1e5 - 200e2), torch.zeros_like(p_full))  # 0 at the surface, 1 at 200 hPa
    new = warmed(control, warming_K + extra_upper_troposphere_warming_K * shape, fixed_rh=keep_relative_humidity)
    o_new = radiation(new)
    budget(o_new, name="warmed")
    dTs = float(new["t_surf"] - control["t_surf"])
    if abs(dTs) < 0.05:
        print("\nwarm the surface (the warming slider) to measure lambda")
    else:
        lam = -float(toa_net(o_new) - toa_net(o_ctl)) / dTs
        print(f"\nfeedback parameter lambda = {lam:.2f} W/m² per K of surface warming")
        print(f"warming this lambda gives for doubled CO₂ (forcing {F_2X:.1f} W/m² at 200 hPa): {F_2X / lam:.2f} K")

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), gridspec_kw=dict(width_ratios=[1, 1, 1.25]))
    axes[0].axvline(0, color=AXIS, linewidth=1)
    axes[0].plot((new["temp"] - control["temp"]).numpy(), p, color=RED)
    axes[0].plot(dTs, 1000, marker="s", markersize=8, color=RED, clip_on=False)
    axes[0].set_xlabel("temperature change (K); square = surface")
    axes[0].set_title("What you changed: temperature")
    axes[1].axvline(0, color=AXIS, linewidth=1)
    axes[1].plot((100 * (new["q"] / control["q"] - 1)).numpy(), p, color=BLUE)
    axes[1].set_xlabel("water vapour change (%)")
    axes[1].set_title("...and water vapour")
    for ax in axes[:2]:
        pressure_axis(ax)
    axes[1].set_ylabel("")
    d_band = (olr_bands(o_new) - olr_bands(o_ctl)).numpy()
    y = np.arange(len(d_band))
    axes[2].barh(y, d_band, height=0.7, color=LW_COLOR)
    axes[2].axvline(0, color=AXIS, linewidth=1)
    axes[2].set_yticks(y, [band_label(lw_spec, b) for b in y])
    axes[2].invert_yaxis()
    axes[2].grid(axis="y", visible=False)
    axes[2].set_xlabel("change in OLR (W/m²)")
    axes[2].set_title("How the OLR responds, band by band")
    fig.tight_layout()
    plt.show()


controls(activity_4, warming_K=slider(1.0, -5.0, 5.0, 0.5, "warming (K)"),
         keep_relative_humidity=tickbox(False, "keep the relative humidity fixed"),
         extra_upper_troposphere_warming_K=slider(0.0, 0.0, 2.0, 0.25, "extra warming of the upper troposphere (K)"));

**Describe how the column responds**
1. Warm the column by 1 K with the water vapour fixed. How much more energy escapes, and from which bands?
   This is the **Planck response**.
2. Now tick *keep the relative humidity fixed*. How does the water vapour change, and at which levels most? Describe
   how λ and the band-by-band response change. This is the **water-vapour feedback**.
3. How much warming from doubled CO₂ does each λ give? By what factor does water vapour amplify it?
4. Add extra upper-tropospheric warming, with and without fixed relative humidity. Describe how λ changes
   in each case. (This is the **lapse-rate feedback**.)
5. Try cooling the column (a negative warming). Is the response the mirror image of warming?

**Explain:** the IPCC's best estimate of the warming from doubled CO₂ is 3 K (likely range 2.5–4 K). Which
processes that this column leaves out could explain the difference?

<details><summary>Hint</summary>

Think about clouds, sea ice and snow, and how the rest of the planet (the poles, the tropics) differs from
this one column.
</details>

*Your notes:*

---
## Activity 5: Which layers matter? Asking the model with gradients

PySoc is written in PyTorch, so it can compute **gradients**: how much an output changes when you nudge
each input. One backward pass gives the sensitivity of an output to the temperature and the water vapour of
*every* layer at once, as if you had perturbed each layer separately. Climate scientists call these
profiles **radiative kernels**. (The original Fortran cannot do this: you would have to perturb the layers
one at a time.)

The plots are per 100 hPa of air, so thin and thick layers can be compared fairly. Choose which output to
examine.

In [ ]:
#@title Activity 5: which layers matter?
OUTPUTS = {"OLR": "soc_olr", "sunlight absorbed by the planet": "soc_toa_sw",
           "longwave from the sky at the surface": "soc_surf_flux_lw_down"}
gradients = {}  # the latest gradients, also used by the check below


def activity_5(output):
    temp = control["temp"].clone().requires_grad_(True)
    t_surf = control["t_surf"].clone().requires_grad_(True)
    log_q = torch.log(control["q"]).clone().requires_grad_(True)
    o = radiation(dict(control, temp=temp, t_surf=t_surf, q=torch.exp(log_q)))
    grads = torch.autograd.grad(o[OUTPUTS[output]], [temp, t_surf, log_q], allow_unused=True)
    d_dT, d_dTs, d_dlogq = (torch.zeros_like(x) if g is None else g for g, x in zip(grads, (temp, t_surf, log_q)))
    gradients.update(output=output, d_dT=d_dT)

    dp = torch.diff(control["p_half"]) / 100  # layer thickness in hPa
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(10.5, 4.4), sharey=True)
    ax.axvline(0, color=AXIS, linewidth=1)
    ax.plot((d_dT / dp * 100).numpy(), p, color=RED)
    ax.set_xlabel("W/m² per K, per 100 hPa of air")
    ax.set_title(f"Sensitivity of the {output} to temperature", fontsize=10)
    ax2.axvline(0, color=AXIS, linewidth=1)
    ax2.plot((d_dlogq / dp * 100 * np.log(1.1)).numpy(), p, color=BLUE)
    ax2.set_xlabel("W/m² for 10% more water vapour, per 100 hPa")
    ax2.set_title("...and to water vapour", fontsize=10)
    pressure_axis(ax)
    fig.tight_layout()
    plt.show()
    above = control["p_full"] < 200e2
    total_T = float(d_dT.sum())
    print(f"warming every layer by 1 K changes the {output} by {total_T:+.2f} W/m²; warming the surface by 1 K: "
          f"{float(d_dTs):+.2f} W/m²")
    if abs(total_T) > 1e-6:
        print(f"share of the air's temperature sensitivity from above 200 hPa: {100 * float(d_dT[above].sum()) / total_T:.0f}%"
              f"  (that air is {100 * float(dp[above].sum()) / 1000:.0f}% of the mass)")
    print(f"10% more water vapour in every layer changes the {output} by {float(d_dlogq.sum()) * np.log(1.1):+.2f} W/m²")


controls(activity_5, output=menu(list(OUTPUTS), "OLR", "output"));

**Check a gradient** by actually warming one layer and rerunning the model. Choose the layer (0 is the top,
39 the bottom) and how much to warm it. The check uses the output chosen above.

In [ ]:
#@title Activity 5: check a gradient by warming one layer
def check_gradient(layer, layer_warming_K):
    key = OUTPUTS[gradients["output"]]
    bump = torch.zeros_like(control["temp"])
    bump[layer] = layer_warming_K
    rerun = float(radiation(dict(control, temp=control["temp"] + bump))[key] - o_ctl[key])
    print(f"layer {layer} at {p[layer]:.1f} hPa, warmed by {layer_warming_K} K; output: {gradients['output']}")
    print(f"  change from rerunning the model:  {rerun:+.5f} W/m²")
    print(f"  change predicted by the gradient: {float(gradients['d_dT'][layer]) * layer_warming_K:+.5f} W/m²")


controls(check_gradient, layer=slider(25, 0, 39, 1, "layer (0 = top, 39 = bottom)"),
         layer_warming_K=slider(1.0, 0.1, 10.0, 0.1, "warming of that layer (K)"));

**Describe what the gradients show**
1. For the OLR: where is the OLR most sensitive to temperature, per 100 hPa? Where does the *total*
   sensitivity come from? How can both be true?
2. For the OLR: where does 10% more water vapour reduce the OLR most? The upper troposphere holds about 100
   times less water than the air near the surface. Per *gram* of added water, which region matters more?
3. Switch the output to the longwave reaching the surface. Describe how the picture changes. Why?
4. Switch to the absorbed sunlight. Which input matters now, and where?
5. Check a few layers with 1 K, 0.1 K and 10 K of warming. Describe how well the gradient predicts the
   change each time.

**Explain:** extra water vapour between about 1 and 30 hPa *increases* the OLR. Why? (Hint: compare the
stratosphere's temperature with the tropopause's.)

*Your notes:*

---
## Activity 6: Add a cloud

Isca's simple cloud scheme passes SOCRATES liquid clouds. Add one to the control column and describe the
response. The change from the clear column is the **cloud radiative effect**: positive at the top means the
cloud warms the planet. (Real high clouds are made of ice; Isca's scheme treats all clouds as liquid.)

A liquid water path of about 10 g/m² is a thin, grey cloud; 200 g/m² is a thick, bright one.

In [ ]:
#@title Activity 6: add a cloud
def activity_6(cloud_top_hPa, cloud_thickness_hPa, liquid_water_path, droplet_radius_microns, cloud_fraction,
               zoom_to_troposphere):
    bottom = min(cloud_top_hPa + cloud_thickness_hPa, 1000)
    cloud = liquid_cloud(control, cloud_top_hPa * 100.0, bottom * 100.0, lwp=liquid_water_path,
                         fraction=cloud_fraction, reff=droplet_radius_microns)
    o_new = radiation(control, cloud=cloud)
    budget(o_new, name="cloudy")
    response_plots(o_new, top=100 if zoom_to_troposphere else 0.01,
                   title=f"A cloud from {cloud_top_hPa} to {bottom} hPa, {liquid_water_path} g/m², fraction {cloud_fraction}")


controls(activity_6, cloud_top_hPa=slider(250, 150, 900, 25, "cloud top (hPa)"),
         cloud_thickness_hPa=slider(100, 50, 300, 25, "cloud thickness (hPa)"),
         liquid_water_path=slider(20, 2, 400, 2, "liquid water path (g/m²)"),
         droplet_radius_microns=slider(10, 4, 30, 1, "droplet radius (µm)"),
         cloud_fraction=slider(1.0, 0.0, 1.0, 0.05, "cloud fraction"),
         zoom_to_troposphere=tickbox(True, "zoom in on the troposphere"));

To see the pattern at a glance, this cell moves a cloud 100 hPa thick up and down, and varies the water in
a high and a low cloud. Change the water path used for the left-hand panel (this takes about a second, so
the plot updates when you let go of the slider).

In [ ]:
#@title Activity 6: move the cloud and change its water
def cloud_effect(cloud_top, lwp, thickness=100.0):
    o = radiation(control, cloud=liquid_cloud(control, cloud_top * 100.0, (cloud_top + thickness) * 100.0, lwp=lwp))
    lw = float(o["soc_olr_clr"] - o["soc_olr"])
    sw = float(o["soc_toa_sw"] - o["soc_toa_sw_clr"])
    return lw, sw, lw + sw


def cloud_sweeps(liquid_water_path_when_moving):
    tops = np.arange(150, 900, 50)
    lwp_values = np.array([2, 5, 10, 20, 50, 100, 200, 400])
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(11, 4.4))
    by_height = np.array([cloud_effect(t, liquid_water_path_when_moving) for t in tops])
    ax.axvline(0, color=AXIS, linewidth=1)
    for i, (name, color) in enumerate((("longwave", LW_COLOR), ("shortwave", SW_COLOR), ("net", NET_COLOR))):
        ax.plot(by_height[:, i], tops, marker="o", markersize=5, color=color, label=name)
    ax.set_ylim(900, 100)
    ax.set_ylabel("cloud-top pressure (hPa)")
    ax.set_xlabel("cloud radiative effect at the top (W/m²)")
    ax.set_title(f"Moving a cloud ({liquid_water_path_when_moving} g/m²) up and down")
    ax.legend(loc="lower right")
    for top, color, name in ((250, BLUE, "high cloud (top 250 hPa)"), (800, AQUA, "low cloud (top 800 hPa)")):
        ax2.plot(lwp_values, [cloud_effect(top, lwp)[2] for lwp in lwp_values], marker="o", markersize=5,
                 color=color, label=name)
    ax2.axhline(0, color=AXIS, linewidth=1)
    ax2.set_xscale("log")
    ax2.set_xticks(lwp_values.tolist(), [str(v) for v in lwp_values])
    ax2.set_xlabel("liquid water path (g/m², log scale)")
    ax2.set_ylabel("net cloud radiative effect (W/m²)")
    ax2.set_title("Thin and thick clouds")
    ax2.legend(loc="lower left")
    fig.tight_layout()
    plt.show()


controls(cloud_sweeps, liquid_water_path_when_moving=slider(50, 2, 400, 2, "liquid water path, left panel (g/m²)",
                                                              live=False));

**Describe how the column responds**
1. Start with the thin high cloud (the default settings). Describe the change in the OLR, in the absorbed
   sunlight, and in the heating inside and below the cloud.
2. Move the same cloud down to 800 hPa. What changes, and what stays about the same?
3. Now make the cloud thick (200 g/m²), high and then low. Which clouds warm the planet overall, and which
   cool it?
4. Change the droplet size with the water fixed. Describe the effect. (Pollution particles make droplets
   smaller: this is one way aerosols affect climate.)
5. Use the second cell to describe how the cloud radiative effect depends on height and on water.

**Explain:** global warming may change how many low clouds there are and how high the high clouds reach.
Using your results, explain why clouds are the biggest source of uncertainty in climate sensitivity.

*Your notes:*

---
## Activity 7: Change the ground: ice, snow and ocean

Sea ice and snow reflect most sunlight (albedo about 0.6–0.8); open ocean absorbs most of it (albedo about
0.06). Change the surface albedo under the clear-sky column (the control uses 0.3), and the height of the
sun. The daily-average sunlight stays at 340 W/m².

In [ ]:
#@title Activity 7: change the surface albedo
albedos = torch.linspace(0.0, 0.9, 10, dtype=torch.float64)
ten_columns = make_column(t_surf=torch.full((len(albedos),), 288.0))


def activity_7(surface_albedo, sun_height_cos_zenith):
    o_ref = radiation(control, coszen=sun_height_cos_zenith)
    o_new = radiation(control, albedo=surface_albedo, coszen=sun_height_cos_zenith)
    budget(o_new, o_ref, name=f"albedo {surface_albedo:.2f}")
    print(f"\nchange in albedo x 340 W/m² = {(0.3 - surface_albedo) * 340:+.1f} W/m² "
          f"(a simple estimate of the change in absorbed sunlight)")
    o_alb = radiation(ten_columns, albedo=albedos, coszen=sun_height_cos_zenith)
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
    ax.plot(albedos.numpy(), o_alb["soc_toa_sw"].numpy(), marker="o", markersize=6, color=SW_COLOR, label="by the planet")
    ax.plot(albedos.numpy(), o_alb["soc_surf_flux_sw"].numpy(), marker="o", markersize=6, color=AQUA, label="at the surface")
    ax.axvline(surface_albedo, color=MUTED, linewidth=1)
    ax.set_xlabel("surface albedo")
    ax.set_ylabel("sunlight absorbed (W/m²)")
    ax.set_title("Absorbed sunlight and surface albedo")
    ax.legend(loc="upper right")
    ax2.axvline(0, color=AXIS, linewidth=1)
    ax2.plot(per_day(o_new["tdt_sw"] - o_ref["tdt_sw"]), p, color=SW_COLOR)
    pressure_axis(ax2, 100)
    ax2.set_xlabel("change in shortwave heating (K/day)")
    ax2.set_title("Change in heating by sunlight")
    fig.tight_layout()
    plt.show()


controls(activity_7, surface_albedo=slider(0.06, 0.0, 0.9, 0.02, "surface albedo (control: 0.3)"),
         sun_height_cos_zenith=slider(0.5, 0.1, 1.0, 0.05, "sun height, cos(zenith angle)"));

**Describe how the column responds**
1. Set the albedo to 0.06 (ocean), then 0.8 (snow). Describe how the sunlight absorbed by the planet, by the
   surface and by the atmosphere changes.
2. Compare the change in absorbed sunlight at the top with the simple estimate (change in albedo × 340).
   Is it larger or smaller?
3. Does the atmosphere's heating change when only the ground changes? Where?
4. Lower the sun. Does the surface albedo matter more or less?

**Explain:** why is the change at the top smaller than the simple estimate? Why is melting ice a *positive*
feedback?

*Your notes:*

---
## Activity 8 (challenge): A model that finds its own temperatures

In 1967, Syukuro Manabe and Richard Wetherald built a one-column model of the atmosphere: radiation plus a
simple rule for convection. They used it to make the first reliable estimate of the warming from doubled
CO₂. Manabe shared the 2021 Nobel Prize in Physics for this line of work. We now have all the pieces to
repeat their experiment with a modern radiation code.

The model steps forward in time, one day per step:
1. radiation heats or cools each layer (`tdt_rad`) and the surface (its net radiation);
2. **convective adjustment**: if the air above the surface is too cold (the temperature falls with height
   faster than a critical lapse rate, 6.5 K/km by default), convection mixes it with the surface into one
   region with exactly that lapse rate, keeping the region's total energy. The region grows upward until
   the air above it is stable;
3. optionally, the water vapour is recomputed to keep the relative humidity fixed.

The surface is a 1 m deep layer of water, and its albedo is 0.23, which gives a surface temperature close
to today's. Run the cell below to define the model (click *Show code* to read it).

In [ ]:
#@title Activity 8: the time-stepping model (run this first)
C_SURF = 4.2e6  # heat capacity of 1 m of water, J/m2 per K


def convective_adjustment(temp, t_surf, p_full, p_half, lapse_rate=6.5):
    """Mix the surface and the unstable air above it to the critical lapse rate (K/km), conserving energy."""
    L = temp.shape[-1]
    # work on numpy arrays of shape (columns, L + 1); the surface is the last "layer"
    T = np.concatenate([temp.numpy(), t_surf.numpy()[..., None]], -1).reshape(-1, L + 1)
    pres = np.concatenate([p_full.numpy(), p_half.numpy()[..., -1:]], -1).reshape(-1, L + 1)
    heat_capacity = np.concatenate([CP_AIR * np.diff(p_half.numpy(), axis=-1) / GRAV,
                                    np.full(temp.shape[:-1] + (1,), C_SURF)], -1).reshape(-1, L + 1)
    # with a constant lapse rate, T(p) = T_surface * (p / p_surface) ** (R * lapse_rate / g)
    shape = (pres / pres[:, -1:]) ** (RDGAS * lapse_rate * 1e-3 / GRAV)
    for i in range(T.shape[0]):
        top, t_base = L, T[i, L]  # the mixed region is levels top..L; it starts as just the surface
        while top > 0 and T[i, top - 1] < t_base * shape[i, top - 1]:  # the air above is too cold: mix it in
            top -= 1
            c = heat_capacity[i, top:]
            t_base = (c * T[i, top:]).sum() / (c * shape[i, top:]).sum()  # keeps the total energy
        T[i, top:] = t_base * shape[i, top:]
    T = T.reshape(temp.shape[:-1] + (L + 1,))
    return torch.from_numpy(T[..., :-1].copy()), torch.from_numpy(T[..., -1].copy())


def equilibrate(col, days=400, convection=True, fixed_rh=True, albedo=0.23, lapse_rate=6.5, rh_surface=0.8,
                insolation=340.0):
    """Step the column forward one day at a time; returns the final column and the net TOA flux each day."""
    col = {k: v.clone() for k, v in col.items()}
    dt = 86400.0
    history = []
    start = time.time()
    with torch.no_grad():
        for day in range(days):
            o = radiation(col, albedo=albedo, insolation=insolation)
            temp = col["temp"] + o["tdt_rad"] * dt
            t_surf = col["t_surf"] + dt * (o["soc_surf_flux_sw"] - o["soc_surf_flux_lw"]) / C_SURF
            if convection:
                temp, t_surf = convective_adjustment(temp, t_surf, col["p_full"], col["p_half"], lapse_rate)
            if not bool(torch.isfinite(temp).all()):
                raise RuntimeError("the temperatures blew up; try less extreme settings")
            col["temp"], col["t_surf"] = temp, t_surf
            col["z_full"], col["z_half"] = hydrostatic_heights(temp, col["p_half"], col["p_full"])
            if fixed_rh:
                col["q"] = specific_humidity(temp, col["p_full"], col["p_half"][..., -1], rh_surface)
            history.append(toa_net(o).numpy().copy())
            if (day + 1) % 100 == 0:
                print(f"day {day + 1}: surface {np.round(t_surf.numpy(), 2)} K, "
                      f"imbalance at the top {np.round(history[-1], 2)} W/m²  ({time.time() - start:.0f} s)")
    return col, np.array(history)


print("Model defined.")

### Part A: radiation alone, then radiation with convection

Start from an atmosphere at one uniform temperature and let it find its own equilibrium, first with
radiation alone and then with convection too. The water vapour stays fixed at the control amounts.
Choose the starting temperature and the critical lapse rate, then press **Run the model** (about 10 s).

In [ ]:
#@title Part A: radiation alone, then with convection
def part_a(starting_temperature_K, critical_lapse_rate_K_per_km):
    start = dict(control, temp=torch.full_like(control["temp"], float(starting_temperature_K)),
                 t_surf=torch.tensor(float(starting_temperature_K), dtype=torch.float64))
    start["z_full"], start["z_half"] = hydrostatic_heights(start["temp"], control["p_half"], control["p_full"])
    print("radiation alone:")
    radiative, hist_a = equilibrate(start, days=500, convection=False, fixed_rh=False)
    print("radiation and convection:")
    rce, hist_b = equilibrate(start, days=300, convection=True, fixed_rh=False, lapse_rate=critical_lapse_rate_K_per_km)

    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(11, 4.5), gridspec_kw=dict(width_ratios=[1.3, 1]))
    for c, color, name in ((radiative, RED, "radiation only"), (rce, INK, "radiation + convection")):
        ax.plot(c["temp"].numpy(), p, color=color, label=name)
        ax.plot(float(c["t_surf"]), 1000, marker="s", markersize=8, color=color, clip_on=False)
    ax.plot(control["temp"].numpy(), p, color=MUTED, linewidth=1.5, label="control column")
    pressure_axis(ax)
    ax.set_xlabel("temperature (K); squares = surface")
    ax.set_title("Equilibrium temperature profiles")
    ax.legend(loc="upper right")
    ax2.axhline(0, color=AXIS, linewidth=1)
    ax2.plot(hist_a, color=RED, label="radiation only")
    ax2.plot(hist_b, color=INK, label="radiation + convection")
    ax2.set_ylim(-30, 40)
    ax2.set_xlabel("day")
    ax2.set_ylabel("net energy in at the top (W/m²)")
    ax2.set_title("Approach to equilibrium")
    ax2.legend(loc="upper right")
    fig.tight_layout()
    plt.show()
    print(f"surface temperature: radiation only {float(radiative['t_surf']):.1f} K, "
          f"with convection {float(rce['t_surf']):.1f} K")


controls(part_a, manual=True, starting_temperature_K=slider(250, 200, 300, 5, "starting temperature (K)"),
             critical_lapse_rate_K_per_km=slider(6.5, 3.0, 9.5, 0.5, "critical lapse rate (K/km)"));

**Describe the equilibria**
1. Describe the equilibrium with radiation alone: the surface temperature, the air just above the surface,
   and how fast the temperature falls with height.
2. Describe how convection changes the profile. Which levels warm and which cool compared with radiation
   alone?
3. Change the starting temperature. Does the equilibrium depend on where it starts? Does the time it takes?
4. Change the critical lapse rate. Describe how the surface temperature and the tropopause respond.

**Explain:** why would the radiation-only atmosphere not survive in reality? What makes the temperature
rise with height in the stratosphere of both equilibria?

*Your notes:*

### Part B: climate sensitivity

Now the real experiment: two columns side by side in one batch, one at 280 ppm and one with more CO₂,
both with convection and fixed relative humidity, run to equilibrium. The difference in their surface
temperatures is this model's **equilibrium climate sensitivity** (for doubled CO₂). Choose the settings and
press **Run the model** (about 5–10 s). If the imbalance at the top has not settled close to zero by the end,
add more days.

In [ ]:
#@title Part B: climate sensitivity
def part_b(co2_ppmv, critical_lapse_rate_K_per_km, surface_relative_humidity, days):
    pair = make_column(t_surf=288.0, co2_ppmv=torch.tensor([280.0, float(co2_ppmv)]), rh_surface=surface_relative_humidity)
    equilibrium, hist_c = equilibrate(pair, days=days, convection=True, fixed_rh=True,
                                      lapse_rate=critical_lapse_rate_K_per_km, rh_surface=surface_relative_humidity)
    warming = float(equilibrium["t_surf"][1] - equilibrium["t_surf"][0])
    print(f"\nsurface temperature at 280 ppm: {float(equilibrium['t_surf'][0]):.2f} K")
    print(f"surface temperature at {co2_ppmv} ppm: {float(equilibrium['t_surf'][1]):.2f} K")
    print(f"surface warming: {warming:+.2f} K")

    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.axvline(0, color=AXIS, linewidth=1)
    ax.plot((equilibrium["temp"][1] - equilibrium["temp"][0]).numpy(), p, color=INK)
    ax.plot(warming, 1000, marker="s", markersize=8, color=INK, clip_on=False)
    pressure_axis(ax)
    ax.set_xlabel("temperature change (K); square = surface")
    ax.set_title(f"Equilibrium response to {co2_ppmv} ppm CO₂")
    fig.tight_layout()
    plt.show()


controls(part_b, manual=True, co2_ppmv=slider(560, 140, 2240, 10, "CO₂ (ppm)"),
             critical_lapse_rate_K_per_km=slider(6.5, 3.0, 9.5, 0.5, "critical lapse rate (K/km)"),
             surface_relative_humidity=slider(0.8, 0.1, 1.0, 0.05, "surface relative humidity"),
             days=slider(400, 100, 1000, 50, "days to run"));

**Describe the equilibrium response**
1. With 560 ppm, describe the change in temperature at the surface, through the troposphere and in the
   stratosphere. Compare with the heating pattern from Activity 3.
2. Compare the surface warming with your λ estimate from Activity 4, and with Manabe and Wetherald's 2.4 K
   (1967, fixed relative humidity).
3. Try 1120 ppm and 140 ppm. Is the warming for each doubling (or halving) about the same?
4. Change the critical lapse rate (for example 9.5 K/km, close to a dry atmosphere) and the relative
   humidity. Describe how the climate sensitivity responds to each.

**Explain:** list the feedbacks this model leaves out. For each, would you expect it to raise or lower the
warming?

*Your notes:*

---
## More to explore

Every activity's code can be changed (click *Show code*). A few more perturbations to try:
* **Ozone hole:** use `make_column(ozone_du=150)` in Activity 1 instead of changing the CO₂. Describe the
  stratosphere's response.
* **Methane:** double it with `model.config.well_mixed[6] *= 2` (gas 6 is CH₄; set it back afterwards).
  Compare its forcing with CO₂'s.
* **Faint young Sun:** 4 billion years ago the Sun was about 25% dimmer. In Part B's code, pass
  `insolation=0.75 * 340` to `equilibrate` and find how much CO₂ keeps the surface near 288 K.
* **A warmer, wetter planet:** repeat Activity 4 with `make_column(t_surf=310, rh_surface=0.9)` as the control.
  Does the water-vapour feedback get stronger?